# 04 — Edge-IIoTset Model Training

**Goal:** Train five classifiers using the full frozen training partition.  
**Input:** Core16 training and validation CSV.GZ datasets.  
**Output:** Models, confusion matrices, metrics and laptop resource measurements.  
**Rule:** Test data is never opened. Edge-IIoTset rows are packet/protocol records, not verified flow records.

In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
from threading import Thread, Event
import hashlib, inspect, json, os, gc, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, psutil, sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score)

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
SPLITS = ROOT / 'data/splits/edge_iiotset'
TRAIN_FILE = SPLITS / 'edge_iiotset_core16_train_v3.csv.gz'
VALID_FILE = SPLITS / 'edge_iiotset_core16_validation_v3.csv.gz'
MANIFEST = SPLITS / 'edge_iiotset_core16_signature_split_manifest_v3.csv'
SPLIT_REPORT = ROOT / 'reports/audits/edge_iiotset_v3_split_report.json'
MODEL_DIR = ROOT / 'models/edge_iiotset_v3'
AUDITS = ROOT / 'reports/audits'
METRICS = ROOT / 'reports/metrics'
FIGURES = ROOT / 'reports/figures'
SEED = 42
MODEL_NAMES = ['decision_tree', 'random_forest', 'extra_trees', 'lightgbm', 'adaboost']
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown', '<missing>'}
SINGLE_FLOW_RUNS = 100  # Samples timing only; models use all training and validation rows.

for path in (TRAIN_FILE, VALID_FILE, MANIFEST, SPLIT_REPORT):
    if not path.is_file():
        raise FileNotFoundError(f'Missing Notebook 03 file: {path}')
report = json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
if report.get('dataset') != 'Edge-IIoTset':
    raise ValueError('Wrong dataset in split report.')
counts = {p: int(report['counts'][p]['rows']) for p in ('train', 'validation', 'test')}
if sum(counts.values()) != 157800 or report.get('total_rows_preserved') != 157800:
    raise ValueError(f'Frozen record counts are incorrect: {counts}')
if report.get('normalized_feature_group_overlap') != 0:
    raise ValueError('Core16 groups overlap between partitions. Do not train.')
if report.get('test_scored') is True:
    raise ValueError('The frozen test partition was already scored; investigate.')
sha = hashlib.sha256()
with MANIFEST.open('rb') as fh:
    for block in iter(lambda: fh.read(8*1024*1024), b''):
        sha.update(block)
if sha.hexdigest() != report.get('manifest_sha256'):
    raise ValueError('Frozen manifest changed after Notebook 03.')
FEATURES = list(report['selected_features'])
CAT = list(report['categorical_features'])
NUM = [f for f in FEATURES if f not in CAT]
if len(FEATURES) != 16 or len(CAT) != 2 or len(NUM) != 14:
    raise ValueError('Core16 feature count/types do not match Notebook 03.')
if len(set(FEATURES)) != 16 or not set(CAT).issubset(FEATURES):
    raise ValueError('Invalid Core16 feature schema.')
if report.get('exported_columns') != ['csv_row_index_0based'] + FEATURES + ['Attack_label','Attack_type']:
    raise ValueError('The frozen report has a different column order.')
for folder in (MODEL_DIR,AUDITS,METRICS,FIGURES):
    folder.mkdir(parents=True,exist_ok=True)
print(f'Edge-IIoTset: {sum(counts.values()):,} total rows | {len(FEATURES)} inputs')
print(f'Train: {counts["train"]:,} | Validation: {counts["validation"]:,} | Test kept sealed')
print('Excel files are for reading only. Models use CSV.GZ data.')

## 1. Load the frozen training and validation partitions

Check row counts, IDs and labels. Do not open the test file.

In [ ]:
FIELDS = ['csv_row_index_0based'] + FEATURES + ['Attack_label', 'Attack_type']
train = pd.read_csv(TRAIN_FILE, dtype=str, keep_default_na=False, low_memory=False)
valid = pd.read_csv(VALID_FILE, dtype=str, keep_default_na=False, low_memory=False)
for part, frame in [('train', train), ('validation', valid)]:
    if frame.columns.tolist() != FIELDS:
        raise ValueError(f'{part}: columns do not match the Core16 export.')
    if len(frame) != counts[part]:
        raise ValueError(f'{part}: expected {counts[part]:,}, found {len(frame):,}.')
    if frame['csv_row_index_0based'].duplicated().any():
        raise ValueError(f'{part}: duplicate original row ID.')
    if not frame['Attack_label'].isin(['0','1']).all():
        raise ValueError(f'{part}: invalid binary label.')
    expected = report['counts'][part]
    if ((frame['Attack_label']=='0').sum() != int(expected['normal']) or
        (frame['Attack_label']=='1').sum() != int(expected['attack'])):
        raise ValueError(f'{part}: class counts differ from Notebook 03.')
if set(train['csv_row_index_0based']).intersection(valid['csv_row_index_0based']):
    raise ValueError('Original row IDs overlap between train and validation.')

coverage = pd.concat([
    train['Attack_type'].value_counts().rename('train'),
    valid['Attack_type'].value_counts().rename('validation')
],axis=1).fillna(0).astype('int64')
missing_validation_types = coverage.index[(coverage['train']>0)&(coverage['validation']==0)].tolist()
if missing_validation_types:
    print('WARNING — no validation records for:', missing_validation_types)
    print('Per-attack-type recall cannot be measured for these types.')
print('Attack-type counts (diagnostic only):')
display(coverage)

y_train = train['Attack_label'].astype(np.uint8).to_numpy()
y_valid = valid['Attack_label'].astype(np.uint8).to_numpy()
X_train_raw = train[FEATURES].copy()
X_valid_raw = valid[FEATURES].copy()
del train, valid
gc.collect()
print(f'All training rows: {len(y_train):,}; all validation rows: {len(y_valid):,}.')
print('Labels and row IDs excluded from features. Test file never opened.')

## 2. Prepare features

Convert invalid numeric values to missing values. Fit imputation and encoding **only on training data**.

In [ ]:
def clean_features(frame):
    out = frame[FEATURES].copy()
    for column in NUM:
        values = out[column].astype(str).str.strip()
        values = values.mask(values.str.casefold().isin(MISSING))
        converted = pd.to_numeric(values,errors='coerce').replace([np.inf,-np.inf],np.nan)
        out[column] = converted.astype(np.float32)
    for column in CAT:
        values = out[column].astype(str).str.strip()
        out[column] = values.mask(values.str.casefold().isin(MISSING),'__MISSING__')
    return out

X_train = clean_features(X_train_raw)
X_valid = clean_features(X_valid_raw)
del X_train_raw,X_valid_raw
gc.collect()
ONE_FLOW = X_valid.iloc[[0]]

def build_preprocessor():
    onehot = {'handle_unknown':'ignore'}
    if 'sparse_output' in inspect.signature(OneHotEncoder).parameters:
        onehot['sparse_output'] = True
    else:
        onehot['sparse'] = True
    impute = {'strategy':'median'}
    if 'keep_empty_features' in inspect.signature(SimpleImputer).parameters:
        impute['keep_empty_features'] = True
    return ColumnTransformer([
        ('numeric', SimpleImputer(**impute), NUM),
        ('category', OneHotEncoder(**onehot), CAT)
    ], sparse_threshold=1.0)

process = psutil.Process(os.getpid())
def rss_mib():
    return process.memory_info().rss/(1024**2)
def cpu_seconds():
    c=process.cpu_times()
    return c.user+c.system

def measure_resources(fn):
    initial=rss_mib()
    peak=[initial]
    stop=Event()
    def monitor():
        while not stop.wait(.02):
            peak[0]=max(peak[0],rss_mib())
    t=Thread(target=monitor,daemon=True)
    c0=cpu_seconds();t0=perf_counter();t.start()
    try:
        result=fn()
    finally:
        seconds=perf_counter()-t0
        c=max(0.0,cpu_seconds()-c0)
        stop.set();t.join()
        peak[0]=max(peak[0],rss_mib())
    return result,{'seconds':round(seconds,4),
                   'process_cpu_pct':round(100*c/max(seconds,1e-9),2),
                   'peak_rss_mib':round(peak[0],2),
                   'rss_increase_mib':round(max(0,peak[0]-initial),2)}

print('Training inputs:',X_train.shape,'| Validation inputs:',X_valid.shape)
print('Numeric zeros are kept; they are not automatically treated as missing.')
print('Process RAM includes data, Python and libraries (not model-only RAM).')

## 3. Shared model evaluation

Train one model, evaluate every validation row, then save its model and results.

In [ ]:
def evaluate_model(name,estimator):
    print('\n===',name.replace('_',' ').title(),'===')
    pipe=Pipeline([('preprocessing',build_preprocessor()),('classifier',estimator)])
    _, training=measure_resources(lambda:pipe.fit(X_train,y_train))
    prediction, predicting=measure_resources(lambda:pipe.predict(X_valid))
    tn,fp,fn,tp=(int(x) for x in confusion_matrix(y_valid,prediction,labels=[0,1]).ravel())

    # Includes preprocessing plus classification for an individual packet record.
    for _ in range(5):
        pipe.predict(ONE_FLOW)
    sample_ids=np.linspace(0,len(X_valid)-1,num=min(SINGLE_FLOW_RUNS,len(X_valid)),dtype=int)
    timings=[]
    for idx in sample_ids:
        start=perf_counter()
        pipe.predict(X_valid.iloc[[int(idx)]])
        timings.append((perf_counter()-start)*1000)

    model_file=MODEL_DIR/f'{name}_core16_v3.joblib'
    joblib.dump(pipe,model_file,compress=3)
    metrics={
        'model':name,'training_rows':int(len(y_train)),
        'validation_rows':int(len(y_valid)),
        'accuracy':round(float(accuracy_score(y_valid,prediction)),6),
        'balanced_accuracy':round(float(balanced_accuracy_score(y_valid,prediction)),6),
        'precision_attack':round(float(precision_score(y_valid,prediction,zero_division=0)),6),
        'recall_attack':round(float(recall_score(y_valid,prediction,zero_division=0)),6),
        'f1_attack':round(float(f1_score(y_valid,prediction,zero_division=0)),6),
        'false_positive_rate':round(fp/max(tn+fp,1),6),
        'false_negative_rate':round(fn/max(tp+fn,1),6),
        'tn':tn,'fp':fp,'fn':fn,'tp':tp
    }
    resources={
        'model':name,'training_seconds':training['seconds'],
        'train_process_cpu_percent':training['process_cpu_pct'],
        'train_peak_process_ram_mib':training['peak_rss_mib'],
        'train_peak_ram_increase_mib':training['rss_increase_mib'],
        'validation_predict_seconds':predicting['seconds'],
        'predict_process_cpu_percent':predicting['process_cpu_pct'],
        'predict_peak_process_ram_mib':predicting['peak_rss_mib'],
        'batch_throughput_records_per_second':round(len(y_valid)/max(predicting['seconds'],1e-9),1),
        'single_record_p50_ms':round(float(np.percentile(timings,50)),3),
        'single_record_p95_ms':round(float(np.percentile(timings,95)),3),
        'single_record_runs':len(timings),
        'model_size_mib':round(model_file.stat().st_size/(1024**2),4)
    }
    clf=pipe.named_steps['classifier']
    top=[]
    if hasattr(clf,'feature_importances_'):
        feature_names=pipe.named_steps['preprocessing'].get_feature_names_out()
        importances=clf.feature_importances_
        top=[{'feature':str(feature_names[i]),'importance':round(float(importances[i]),6)}
             for i in np.argsort(importances)[::-1][:10]]
    artifact={
        'experiment':'Edge-IIoTset Core16 V3 | validation only',
        'created_utc':datetime.now(timezone.utc).isoformat(),
        'model':name,'features':FEATURES,'target':'Attack_label',
        'label_mapping':{'0':'Normal','1':'Attack'},'seed':SEED,
        'frozen_manifest_sha256':report['manifest_sha256'],
        'model_file':model_file.name,'metrics':metrics,'resources':resources,
        'top_transformed_feature_importance':top,
        'attack_types_absent_from_validation':missing_validation_types,
        'versions':{'python':platform.python_version(),'scikit_learn':sklearn.__version__},
        'test_scored':False,
        'limitations':[
            'Edge-IIoTset selected ML data are packet/protocol records, not verified flows.',
            'No per-type recall can be measured for categories missing from validation.',
            'Exact feature-group separation does not prove capture/session independence.',
            'Mixed-label feature patterns make some Core16 records indistinguishable.',
            'High zero rates may encode protocol applicability or dataset shortcuts.',
            'Laptop CPU, RAM and latency are not Raspberry Pi or end-to-end network timing.',
            'Feature importance is estimator-specific and is not causal evidence.'
        ]
    }
    audit_path=AUDITS/f'edge_iiotset_v3_{name}_training.json'
    audit_path.write_text(json.dumps(artifact,indent=2),encoding='utf-8')
    fig,ax=plt.subplots(figsize=(5.5,4.4))
    ConfusionMatrixDisplay.from_predictions(y_valid,prediction,labels=[0,1],
        display_labels=['Normal','Attack'],cmap='Blues',values_format='d',ax=ax)
    ax.set_title(f'{name.replace("_"," ").title()} — Edge-IIoTset validation')
    fig.tight_layout()
    fig.savefig(FIGURES/f'edge_iiotset_v3_{name}_confusion_matrix.png',dpi=170,bbox_inches='tight')
    plt.show();plt.close(fig)
    display(pd.DataFrame([metrics]).T.rename(columns={0:name}))
    display(pd.DataFrame([resources]).T.rename(columns={0:name}))
    if top: display(pd.DataFrame(top))
    print('Saved:',model_file.name,'|',audit_path.name)
    return artifact

## 4. Decision Tree

A small single-tree baseline.

In [ ]:
decision_tree_result=evaluate_model('decision_tree',
    DecisionTreeClassifier(max_depth=14,min_samples_leaf=30,
                           class_weight='balanced',random_state=SEED))

## 5. Random Forest

Compare accuracy with CPU and model size.

In [ ]:
random_forest_result=evaluate_model('random_forest',
    RandomForestClassifier(n_estimators=100,max_depth=16,min_samples_leaf=20,
                           class_weight='balanced_subsample',random_state=SEED,n_jobs=-1))

## 6. Extra Trees

A randomized tree ensemble.

In [ ]:
extra_trees_result=evaluate_model('extra_trees',
    ExtraTreesClassifier(n_estimators=100,max_depth=16,min_samples_leaf=20,
                         class_weight='balanced',random_state=SEED,n_jobs=-1))

## 7. LightGBM

Boosted trees; multicore CPU can exceed 100%.

In [ ]:
from lightgbm import LGBMClassifier
lightgbm_result=evaluate_model('lightgbm',
    LGBMClassifier(n_estimators=160,num_leaves=31,learning_rate=.05,
                   min_child_samples=30,random_state=SEED,n_jobs=4,verbosity=-1))

## 8. AdaBoost

A sequence of shallow decision trees.

In [ ]:
weak=DecisionTreeClassifier(max_depth=2,min_samples_leaf=30,random_state=SEED)
ada_args={'n_estimators':100,'learning_rate':.5,'random_state':SEED}
if 'estimator' in inspect.signature(AdaBoostClassifier).parameters:
    ada_args['estimator']=weak
else:
    ada_args['base_estimator']=weak
adaboost_result=evaluate_model('adaboost',AdaBoostClassifier(**ada_args))

## 9. Compare five models

Compare validation performance, resource cost and saved model sizes.

In [ ]:
results=[]
for name in MODEL_NAMES:
    path=AUDITS/f'edge_iiotset_v3_{name}_training.json'
    if not path.is_file():
        raise FileNotFoundError(f'Missing model report: {path.name}. Run its training cell.')
    record=json.loads(path.read_text(encoding='utf-8'))
    if record.get('frozen_manifest_sha256')!=report['manifest_sha256']:
        raise ValueError(f'{name}: saved results belong to a different frozen split.')
    if (record['metrics']['training_rows']!=counts['train'] or
        record['metrics']['validation_rows']!=counts['validation']):
        raise ValueError(f'{name}: incomplete or mismatched training records.')
    results.append(record)
comparison=pd.DataFrame([r['metrics'] for r in results])
resources=pd.DataFrame([r['resources'] for r in results])
comparison.to_csv(METRICS/'edge_iiotset_v3_five_model_validation.csv',index=False)
resources.to_csv(METRICS/'edge_iiotset_v3_five_model_resources.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(12,4.6))
labels=comparison['model'].str.replace('_',' ').str.title()
axes[0].bar(labels,comparison['f1_attack']*100)
axes[0].set_title('Edge-IIoTset — Attack F1 (validation)')
axes[0].set_ylabel('F1-score (%)');axes[0].set_ylim(0,105)
axes[1].bar(labels,comparison['false_positive_rate']*100)
axes[1].set_title('Edge-IIoTset — False positive rate (validation)')
axes[1].set_ylabel('False-positive rate (%)')
for ax in axes:
    ax.tick_params(axis='x',labelrotation=17)
    ax.grid(axis='y',alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES/'edge_iiotset_v3_model_detection.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)

fig,axes=plt.subplots(2,2,figsize=(11,8))
for ax,(field,title,unit) in zip(axes.flat,[
    ('training_seconds','Training time','seconds'),
    ('train_peak_process_ram_mib','Peak training process RAM','MiB'),
    ('single_record_p95_ms','Single-record prediction p95','ms'),
    ('model_size_mib','Saved model size','MiB')]):
    ax.bar(resources['model'].str.replace('_',' ').str.title(),resources[field])
    ax.set_title(title);ax.set_ylabel(unit)
    ax.tick_params(axis='x',labelrotation=17)
    ax.grid(axis='y',alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES/'edge_iiotset_v3_model_resources.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)

display(comparison[['model','accuracy','precision_attack','recall_attack','f1_attack',
                    'false_positive_rate','false_negative_rate','tn','fp','fn','tp']])
display(resources[['model','training_seconds','train_process_cpu_percent',
                    'train_peak_process_ram_mib','validation_predict_seconds',
                    'batch_throughput_records_per_second','single_record_p95_ms','model_size_mib']])
print('Saved: five models, five individual reports, five confusion matrices,')
print('two comparison CSVs and two charts. Test set has NOT been opened.')
if missing_validation_types:
    print('WARNING: per-type validation not possible for:',missing_validation_types)